# Electricity Theft Detection — Full Benchmark (Colab GPU, BACKUP)

Use this only if Kaggle quota/session fails. Same pipeline, same output.

**Before running:**
1. Upload `etd-repo.zip` (built by `make_etd_repo_zip.sh`) to your **Google Drive root**.
2. Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# Cell 1 — dependencies + GPU check
!pip install -q imbalanced-learn openpyxl
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# Cell 2 — pull the project zip from Google Drive and unzip
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/run
!cp "/content/drive/MyDrive/etd-repo.zip" /content/etd-repo.zip
!unzip -q /content/etd-repo.zip -d /content/run
%cd /content/run
!ls

In [ ]:
# Cell 3 — convert Maheen's Pakistan workbook into the Stage 5 target CSV.
# Expected end line: Class balance (0=normal, 1=theft): {0: 42, 1: 42}
!python -m src.experiments.prepare_pakistan_target --input pakistan_sgcc_format.xlsx

In [ ]:
# Cell 4 — the FULL benchmark: preprocess -> stage1 -> stage2 -> stage3 -> stage4 -> stage5
# If this cell dies mid-run, re-run Cell 2, then add e.g. --from_stage 3 to resume.
!python -m src.experiments.run_all_benchmark --config config/config.yaml \
    --target_csv data/raw/pakistan/pakistan_target.csv

In [ ]:
# Cell 5 — package and download trained weights + results
!zip -qr /content/results_and_models.zip experiments_results models
from google.colab import files
files.download('/content/results_and_models.zip')

## Bring results back to the laptop
```bash
cd electricity-theft-detection
unzip -o ~/Downloads/results_and_models.zip
```
This merges `models/checkpoints/` (API/demo weights), `models/stage_checkpoints/` (per-stage models) and `experiments_results/benchmark_results.json` (the final table) into the project.